## Camera Lidar Calibration Example

The following notebooks serves as an example for calibrating a Camera and a Lidar. The example remains as ROS2 agnostic as possible. Converting the code to a ROS2 node remains as an exercize for the students.
This example uses the publically available KITTI dataset more information can be found [here](https://www.cvlibs.net/datasets/kitti/)

First download the dataset

In [ ]:
from pathlib import Path
from glob import glob
import os, zipfile, urllib.request, textwrap, math
import numpy as np
import cv2
import matplotlib.pyplot as plt
import plotly.graph_objects as go



In [ ]:

plt.rcParams["figure.figsize"] = (16, 9)
np.set_printoptions(precision=4, suppress=True)

DATA_ROOT = Path("kitti_raw")
DATA_ROOT.mkdir(exist_ok=True)

DRIVE_URL = "https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data/2011_10_03_drive_0047/2011_10_03_drive_0047_sync.zip"
CALIB_URL = "https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data/2011_10_03_calib.zip"

DOWNLOAD_FULL_SEQUENCE = True 

def download(url, out_path):
    out_path = Path(out_path)
    if out_path.exists():
        print(f"Already exists: {out_path}")
        return out_path
    print(f"Downloading {url}\n -> {out_path}")
    urllib.request.urlretrieve(url, out_path)
    return out_path

def unzip(zip_path, dest):
    zip_path, dest = Path(zip_path), Path(dest)
    marker = dest / (zip_path.stem + ".unzipped")
    if marker.exists():
        print(f"Already unzipped: {zip_path.name}")
        return
    print(f"Unzipping {zip_path.name} ...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(dest)
    marker.write_text("done")
    print("Done.")

if DOWNLOAD_FULL_SEQUENCE:
    drive_zip = download(DRIVE_URL, DATA_ROOT / "2011_10_03_drive_0047_sync.zip")
    calib_zip = download(CALIB_URL, DATA_ROOT / "2011_10_03_calib.zip")
    unzip(drive_zip, DATA_ROOT)
    unzip(calib_zip, DATA_ROOT)
else:
    print("Manual mode: place the extracted KITTI folders under:", DATA_ROOT.resolve())


Select camera and LiDAR frame and load the images and point clouds

In [ ]:
DRIVE = DATA_ROOT / "2011_10_03" / "2011_10_03_drive_0047_sync"
CALIB_DIR = DATA_ROOT / "2011_10_03"

image_paths = sorted((DRIVE / "image_02" / "data").glob("*.png"))
velo_paths  = sorted((DRIVE / "velodyne_points" / "data").glob("*.bin"))

print("Images:", len(image_paths))
print("LiDAR scans:", len(velo_paths))
assert len(image_paths) > 0 and len(velo_paths) > 0, "Dataset not found. Check DATA_ROOT and extraction."

FRAME_ID = 0
img_path = image_paths[FRAME_ID]
velo_path = velo_paths[FRAME_ID]
print("Using image:", img_path)
print("Using LiDAR:", velo_path)

Intrinsic camera intrisics parameters are needed for proper calibration, it has been demonstrated in previous notebooks how to obtain them as such they are given here. 

In [ ]:
def read_kitti_calib_file(path):
    data = {}
    with open(path, "r") as f:
        for line in f:
            if ":" not in line:
                continue
            key, value = line.split(":", 1)
            value = value.strip()
            if len(value) == 0:
                continue
            try:
                data[key] = np.array([float(x) for x in value.split()])
            except ValueError:
                data[key] = value
    return data

cam_calib = read_kitti_calib_file(CALIB_DIR / "calib_cam_to_cam.txt")
velo_calib = read_kitti_calib_file(CALIB_DIR / "calib_velo_to_cam.txt")

# Camera 02 projection matrix after rectification.
P_rect_02 = cam_calib["P_rect_02"].reshape(3, 4)
K = P_rect_02[:3, :3]

# Rectification rotation.
R_rect_00 = np.eye(4)
R_rect_00[:3, :3] = cam_calib["R_rect_00"].reshape(3, 3)

# Velodyne -> reference camera transform.
T_velo_to_cam = np.eye(4)
T_velo_to_cam[:3, :3] = velo_calib["R"].reshape(3, 3)
T_velo_to_cam[:3, 3] = velo_calib["T"]

# Velodyne -> rectified camera 02 coordinates.
T_velo_to_cam_rect = R_rect_00 @ T_velo_to_cam

print("K =\n", K)
print("\nT_velo_to_cam_rect =\n", T_velo_to_cam_rect)


Before calibrating and projecting the sensors it is important to inspect the data. The following code visualizes camera photos and lidar point clouds. Feel free to visualizes different frames by changing the code below.

In [ ]:

def load_velodyne_bin(path):
    points = np.fromfile(str(path), dtype=np.float32).reshape(-1, 4)
    return points

img_path = "kitti_raw/2011_10_03/2011_10_03_drive_0047_sync/image_02/data/0000000700.png"
velo_path = "kitti_raw/2011_10_03/2011_10_03_drive_0047_sync/velodyne_points/data/0000000700.bin"

# Load image

image_bgr = cv2.imread(str(img_path))
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

# Load LiDAR
points_velo = load_velodyne_bin(velo_path)

print("Image shape:", image_rgb.shape)
print("Point cloud shape:", points_velo.shape)

# SHOW CAMERA IMAGE
plt.figure(figsize=(14, 8))
plt.imshow(image_rgb)
plt.axis("off")
plt.title("Camera Image")
plt.show()

# PREPARE POINT CLOUD

xyz = points_velo[:, :3]

x = xyz[:, 0]
y = xyz[:, 1]
z = xyz[:, 2]

# Keep only points in front of vehicle
mask = x > 0

x = x[mask]
y = y[mask]
z = z[mask]

# Optional downsampling for performance
N = 50000

if len(x) > N:
    idx = np.random.choice(len(x), N, replace=False)
    x = x[idx]
    y = y[idx]
    z = z[idx]

# Distance from sensor
distance = np.sqrt(x**2 + y**2 + z**2)


fig = go.Figure(
    data=[
        go.Scatter3d(
            x=x,
            y=y,
            z=z,
            mode='markers',
            marker=dict(
                size=1.5,
                color=distance,
                colorscale='Jet',
                opacity=0.8,
                colorbar=dict(title="Distance")
            )
        )
    ]
)

fig.update_layout(
    title="Interactive LiDAR Point Cloud",
    scene=dict(
        xaxis_title="X (Forward)",
        yaxis_title="Y (Left/Right)",
        zaxis_title="Z (Height)",
        aspectmode='data'
    ),
    height=800,
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()

## Projecting LiDAR Points into the Camera Image

This code projects 3D LiDAR points into 2D image coordinates using the camera intrinsics `K` and the LiDAR-to-camera transformation matrix `T_velo_to_cam_rect`.

First, LiDAR points are converted to homogeneous coordinates:

$$
\tilde{p}_{velo} =
\begin{bmatrix}
x \\
y \\
z \\
1
\end{bmatrix}
$$

Then, they are transformed from the LiDAR frame into the camera frame:

$$
\tilde{p}_{cam} =
T_{velo \rightarrow cam}
\tilde{p}_{velo}
$$

which applies a rigid transformation:

$$
p_{cam} = R p_{velo} + t
$$

Only points with positive depth are kept:

$$
Z_c > 0
$$

The camera intrinsic matrix

$$
K =
\begin{bmatrix}
f_x & 0 & c_x \\
0 & f_y & c_y \\
0 & 0 & 1
\end{bmatrix}
$$

is then used to project 3D camera points into image coordinates:

$$
\begin{bmatrix}
u' \\
v' \\
w'
\end{bmatrix}
=
K
\begin{bmatrix}
X_c \\
Y_c \\
Z_c
\end{bmatrix}
$$

Pixel coordinates are obtained by perspective division:

$$
u = \frac{u'}{w'}, \qquad
v = \frac{v'}{w'}
$$

Finally, only projected points that fall inside the image boundaries are kept.

In [ ]:
def project_lidar_to_image(points_velo_xyz, K, T_velo_to_cam_rect, image_shape):
    h, w = image_shape[:2]
    pts_h = np.hstack([points_velo_xyz, np.ones((len(points_velo_xyz), 1))])
    pts_cam = (T_velo_to_cam_rect @ pts_h.T).T[:, :3]

    # Keep points in front of camera.
    in_front = pts_cam[:, 2] > 0.1
    pts_cam = pts_cam[in_front]
    pts_velo_kept = points_velo_xyz[in_front]

    uvw = (K @ pts_cam.T).T
    uv = uvw[:, :2] / uvw[:, 2:3]

    in_img = (
        (uv[:, 0] >= 0) & (uv[:, 0] < w) &
        (uv[:, 1] >= 0) & (uv[:, 1] < h)
    )
    return uv[in_img], pts_cam[in_img], pts_velo_kept[in_img]

uv_gt, pts_cam_gt, pts_velo_visible = project_lidar_to_image(
    points_velo[:, :3], K, T_velo_to_cam_rect, image_rgb.shape
)

print("Visible projected LiDAR points:", len(uv_gt))
print("Example uv:", uv_gt[:5])


In [ ]:
def draw_projected_points(image_rgb, uv, depth, max_points=20000):
    out = image_rgb.copy()
    if len(uv) > max_points:
        idx = np.random.default_rng(0).choice(len(uv), max_points, replace=False)
        uv, depth = uv[idx], depth[idx]

    d = np.clip(depth, 0, 80)
    d_norm = (255 * (1 - d / 80)).astype(np.uint8)
    colors = cv2.applyColorMap(d_norm.reshape(-1, 1), cv2.COLORMAP_JET).reshape(-1, 3)

    for (u, v), c in zip(uv.astype(int), colors):
        cv2.circle(out, (int(u), int(v)), 1, tuple(int(x) for x in c.tolist()), -1)
    return out

overlay_gt = draw_projected_points(image_rgb, uv_gt, pts_cam_gt[:, 2])
plt.imshow(overlay_gt)
plt.axis("off")
plt.title("KITTI ground-truth camera–LiDAR projection")
plt.show()

Simulate uncertainty

In [ ]:
rng = np.random.default_rng(42)

N_CORR = 300
idx = rng.choice(len(pts_velo_visible), size=N_CORR, replace=False)

object_points_lidar = pts_velo_visible[idx].astype(np.float32)   # 3D points in LiDAR frame
image_points = uv_gt[idx].astype(np.float32)                    # 2D pixels from GT projection

PIXEL_NOISE_STD = 1.0
image_points_noisy = image_points + rng.normal(0, PIXEL_NOISE_STD, image_points.shape).astype(np.float32)

print("3D object points:", object_points_lidar.shape)
print("2D image points:", image_points_noisy.shape)

In order to estimate the transform between the LiDAR and camera frames the Perspective-n-Point (PnP) agorithm is used. More information on this algorithm can be found [here](https://en.wikipedia.org/wiki/Perspective-n-Point). The opencv usage can be found here [here](https://docs.opencv.org/4.x/d5/d1f/calib3d_solvePnP.html).

In [ ]:
dist_coeffs = np.zeros((4, 1), dtype=np.float64)

ok, rvec_est, tvec_est, inliers = cv2.solvePnPRansac(
    objectPoints=object_points_lidar,
    imagePoints=image_points_noisy,
    cameraMatrix=K.astype(np.float64),
    distCoeffs=dist_coeffs,
    iterationsCount=200,
    reprojectionError=3.0,
    confidence=0.999,
    flags=cv2.SOLVEPNP_ITERATIVE
)

assert ok, "PnP failed."

R_est, _ = cv2.Rodrigues(rvec_est)
T_est = np.eye(4)
T_est[:3, :3] = R_est
T_est[:3, 3] = tvec_est.ravel()

print("Estimated T_velo_to_cam_rect =\n", T_est)
print("\nKITTI ground-truth T_velo_to_cam_rect =\n", T_velo_to_cam_rect)
print("\nInliers:", None if inliers is None else len(inliers), "/", N_CORR)

In [ ]:
def rotation_error_deg(R1, R2):
    R = R1 @ R2.T
    cos_angle = (np.trace(R) - 1) / 2
    cos_angle = np.clip(cos_angle, -1, 1)
    return np.degrees(np.arccos(cos_angle))

rot_err = rotation_error_deg(T_est[:3, :3], T_velo_to_cam_rect[:3, :3])
trans_err = np.linalg.norm(T_est[:3, 3] - T_velo_to_cam_rect[:3, 3])

print(f"Rotation error: {rot_err:.4f} degrees")
print(f"Translation error: {trans_err:.4f} meters")

# Reprojection error on correspondences
proj, _ = cv2.projectPoints(
    object_points_lidar,
    rvec_est,
    tvec_est,
    K.astype(np.float64),
    dist_coeffs
)
proj = proj.reshape(-1, 2)
reproj_err = np.linalg.norm(proj - image_points, axis=1)
print(f"Mean reprojection error vs clean GT pixels: {reproj_err.mean():.3f} px")
print(f"Median reprojection error: {np.median(reproj_err):.3f} px")


Now we can project the lidar points onto the image.

In [ ]:
uv_est, pts_cam_est, _ = project_lidar_to_image(
    points_velo[:, :3], K, T_est, image_rgb.shape
)

overlay_est = draw_projected_points(image_rgb, uv_est, pts_cam_est[:, 2])
plt.imshow(overlay_est)
plt.axis("off")
plt.title("Estimated camera–LiDAR projection")
plt.show()

In [ ]:

# Project LiDAR points into image using estimated calibration
uv_est, pts_cam_est, pts_velo_visible_est = project_lidar_to_image(
    points_velo[:, :3], K, T_est, image_rgb.shape
)

# Convert projected pixels to integer image coordinates
u = uv_est[:, 0].astype(int)
v = uv_est[:, 1].astype(int)

# Sample RGB color from camera image at each projected pixel
colors_rgb = image_rgb[v, u]   # shape: (N, 3)

# Convert RGB to Plotly color strings
colors_plotly = [
    f"rgb({r},{g},{b})" for r, g, b in colors_rgb
]

# Original LiDAR-frame XYZ points
x = pts_velo_visible_est[:, 0]
y = pts_velo_visible_est[:, 1]
z = pts_velo_visible_est[:, 2]

# Optional downsample for speed
N = 50000
if len(x) > N:
    idx = np.random.choice(len(x), N, replace=False)
    x = x[idx]
    y = y[idx]
    z = z[idx]
    colors_plotly = [colors_plotly[i] for i in idx]

# Plot colored point cloud
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=x,
            y=y,
            z=z,
            mode="markers",
            marker=dict(
                size=1.5,
                color=colors_plotly,
                opacity=0.9
            )
        )
    ]
)

fig.update_layout(
    title="LiDAR Point Cloud Colored by Camera Pixels",
    scene=dict(
        xaxis_title="X LiDAR (Forward)",
        yaxis_title="Y LiDAR (Left/Right)",
        zaxis_title="Z LiDAR (Height)",
        aspectmode="data"
    ),
    height=800,
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()